In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

import umap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
import matplotlib.colors as mcolors
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

/private/tmp/claude-501/-Users-danielleknutson-MIT-Dropbox-Danielle-Knutson-Dataset-project-DER-data-UROP/bb46b6cd-7152-47fc-a949-36d3b6292711/scratchpad/auditenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analysis_df = pd.read_csv("../data/processed/combined_der_dataset_w_controls_predictors.csv")
analysis_df["zip_code"] = analysis_df["zip_code"].astype(str).str.zfill(5)

# Restrict to the SAME sample the regressions use. Previously this notebook clustered
# the full unfiltered file and handed the gaps to SimpleImputer(strategy='median'),
# which meant ~800 ZIPs with no ACS data at all were pushed to the centroid of every
# predictor and duly grouped together: the largest reported cluster was 69% rows with
# no data. That is an imputation artifact, not a socio-technical typology. It also
# left the cluster table (N=2,552) and the summary-statistics table (N=1,386)
# describing different universes.
_n_before = len(analysis_df)
analysis_df["total_population"] = pd.to_numeric(analysis_df["total_population"], errors="coerce")
analysis_df = analysis_df[
    analysis_df["total_population"].notna() & (analysis_df["total_population"] >= 1000)
]
_core_needed = ["median_household_income", "pct_black", "pct_hispanic", "pct_asian"]
analysis_df = analysis_df.dropna(
    subset=[c for c in _core_needed if c in analysis_df.columns]
).reset_index(drop=True)
print(f"Clustering sample: {len(analysis_df)} ZIPs (from {_n_before}); matches the regression sample.")

df = analysis_df.drop(columns=['zip_code','overlap_area_m2','overlap_share_of_zip',
                      'utility','acronym','Unnamed: 0', 'lat', 'lon',
                      't2m_max_mean_c_2023', 't2m_min_mean_c_2023',
                      't2m_summer_mean_c_2023', "Unnamed: 0.1", 'pop_density_km2',
                      'kwh_annual_total', 'county_geoid', 'county_name', 'plant_capacity_mw'],
            errors='ignore').copy()
print(df.columns)

Clustering sample: 1386 ZIPs (from 2552); matches the regression sample.
Index(['PV_system_size_DC', 'total_chargers', 'level1_chargers',
       'level2_chargers', 'dc_fast_chargers', 'zev_count',
       'storage_capacity_mw', 'wind_capacity_mw', 'wind_turbine_count',
       'median_household_income', 'poverty_rate', 'pct_bachelors_plus',
       'pct_black', 'pct_hispanic', 'pct_asian', 'median_housing_value',
       'pct_single_family_units', 'pct_multifamily_units',
       'pct_mobile_home_units', 'owner_occupied_rate', 'total_population',
       'ghi_mean_kwh_m2_day_2023', 't2m_mean_c_2023', 'cdd65_2023',
       'hdd65_2023', 'wind_ws10m_mean_2023', 'wind_ws50m_mean_2023',
       'utility_type', 'log_kwh', 'energy_burden_pct',
       'energy_affordability_index', 'energy_affordability_gap', 'area_km2',
       'log_pop_density'],
      dtype='object')


In [3]:
# safer missing handling
df["utility_type"] = df["utility_type"].fillna("POU")

population = df["total_population"].replace(0, np.nan)

rate_cols = {
    'PV_system_size_DC': ('pv_kw_per_1k', 1000.0),
    'total_chargers': ('chargers_per_1k', 1000.0),
    'zev_count': ('zev_per_1k', 1000.0),
    'storage_capacity_mw': ('storage_mw_per_100k', 100000.0),
    'wind_capacity_mw': ('wind_mw_per_100k', 100000.0),
    'wind_turbine_count': ('turbines_per_100k', 100000.0),
}

for col, (new_name, denom) in rate_cols.items():
    if col in df.columns:
        df[new_name] = df[col] / (population / denom)
        df.drop(columns=[col], inplace=True)

df.replace([np.inf, -np.inf], np.nan, inplace=True)

cluster_columns = ['UMAP_Cluster','PCA_Cluster','tSNE_Cluster',
'Cluster','RF_Cluster', 'cluster_z_pca', "cluster_z2_tsne", "cluster_z2_umap"]
df = df.drop(columns=[c for c in cluster_columns if c in df.columns], errors='ignore')
df = df.dropna(axis=1, how="all")

categorical_cols = ["utility_type"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[('cat', encoder, categorical_cols)],
    remainder='passthrough'
)

df_encoded = preprocessor.fit_transform(df)
df_encoded = pd.DataFrame(df_encoded, columns=preprocessor.get_feature_names_out())

# single imputation
imputer = SimpleImputer(strategy='median')
data_imputed = imputer.fit_transform(df_encoded)

scaler = StandardScaler()
clustering_data_scaled = scaler.fit_transform(data_imputed)
clustering_data_scaled = pd.DataFrame(clustering_data_scaled, columns=df_encoded.columns)

rename_map = {}
for column in clustering_data_scaled.columns:
    if column[:3] == "cat":
        rename_map[column] = column[5:]
    else:
        rename_map[column] = column[11:]
clustering_data_scaled.rename(columns = rename_map, inplace = True)
stand_rename_map = {
    "cdd65_2023": "Cooling degree days",
    "stand_cdd65_2023": "Standardizde cooling degree days",
    "stand_hdd65_2023": "Standardizde heating degree days",
    "hdd65_2023": "Heating degree days",
    "ghi_mean_kwh_m2_day_2023": "Solar irradiance (GHI)",
    "wind_ws10m_mean_2023": "Wind speed (10m)",
    "wind_ws50m_mean_2023": "Wind speed (50m)",
    "poverty_rate": "Poverty rate",
    "pct_bachelors_plus": "% Bachelor's+",
    "pct_black": "% Black",
    "pct_hispanic": "% Hispanic",
    "pct_asian": "% Asian",
    "median_household_income": "Median household income",
    "log_median_housing_value": "Log median housing value",
    "median_housing_value": "Median housing value",
    "log_pop_density": "Log population density",
    "log_kwh": "Log annual electricity demand",
    "pv_kw_per_1k": "Solar PV adoption per 1k pop",
    "chargers_per_1k": "EV charger per 1k pop",
    "zev_per_1k": "EV cars per 1k pop",
    "storage_mw_per_100k": "Storage deployment per 1k pop",
    "wind_mw_per_100k": "Wind MW per 100k",
    "turbines_per_100k": "Number of turbines per 100k",
    "utility_type_IOU": "IOU Utility",
    "utility_type_POU": "POU Utility",
    "level1_chargers": "Number of Level 1 chargers",
    "level2_chargers": "Number of Level 2 chargers",
    "dc_fast_chargers": "Number of DC fast chargers",
    "utility_type": "Utility type",
    "total_population": "Total population",
    "area_km2": "Area in km2",
    "t2m_mean_c_2023": "Temp at 2m above ground",
    "ghi_mean_kwh_m2_day_2024": "Global horizontal irradiance",
    "wind_ws50m_mean_2024": "Mean wind speed at 50 m",
}
clustering_data_scaled.rename(columns = stand_rename_map, inplace = True)
print(clustering_data_scaled.describe())
print(clustering_data_scaled.columns)

        IOU Utility   POU Utility  Number of Level 1 chargers  \
count  1.386000e+03  1.386000e+03                1.386000e+03   
mean  -1.230377e-16  4.101257e-17                5.126571e-18   
std    1.000361e+00  1.000361e+00                1.000361e+00   
min   -3.605551e+00 -2.773501e-01               -8.268063e-02   
25%    2.773501e-01 -2.773501e-01               -8.268063e-02   
50%    2.773501e-01 -2.773501e-01               -8.268063e-02   
75%    2.773501e-01 -2.773501e-01               -8.268063e-02   
max    2.773501e-01  3.605551e+00                2.815883e+01   

       Number of Level 2 chargers  Number of DC fast chargers  \
count                1.386000e+03                1.386000e+03   
mean                 2.563285e-18                3.075943e-17   
std                  1.000361e+00                1.000361e+00   
min                 -4.923257e-01               -5.858210e-01   
25%                 -4.571858e-01               -5.858210e-01   
50%                 -3.1

In [4]:
X = clustering_data_scaled.drop(columns=cluster_columns, errors="ignore")


def set_up_pca_kmeans(n_components, n_clusters, random_state=77):
    pca = PCA(n_components=n_components, random_state=random_state)
    Z_pca = pca.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    labels = kmeans.fit_predict(Z_pca)

    df_viz_pca = X.copy()
    df_viz_pca["cluster_z_pca"] = labels.astype(int)

    print(df_viz_pca["cluster_z_pca"].value_counts().sort_index())

    return {
        "pca": pca,
        "embedding": Z_pca,
        "labels": labels.astype(int),
        "kmeans": kmeans,
        "df_viz": df_viz_pca,
    }

In [5]:
def plot_clusters(Z, labels, centroids=None, title=None):
    plot_df = pd.DataFrame(Z[:, :2], columns=["PC1", "PC2"])
    plot_df["cluster"] = pd.Categorical(labels.astype(int))

    n_clusters = plot_df["cluster"].nunique()
    palette = sns.color_palette("tab10", n_colors=n_clusters)

    plt.figure(figsize=(9, 6))
    ax = sns.scatterplot(
        data=plot_df,
        x="PC1",
        y="PC2",
        hue="cluster",
        palette=palette,
        alpha=0.65,
        s=42,
        linewidth=0.2,
        edgecolor="white"
    )

    if centroids is not None:
        centroids_2d = centroids[:, :2]
        ax.scatter(
            centroids_2d[:, 0],
            centroids_2d[:, 1],
            marker="X",
            s=240,
            c="black",
            linewidth=1.0,
            edgecolor="white",
            label="Centroids"
        )

    # Overall title omitted; the figure caption carries it.
    ax.set_xlabel("Principal component 1")
    ax.set_ylabel("Principal component 2")
    ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
    sns.despine()
    plt.tight_layout()
    plt.show()

In [6]:
"""
Optional clustering methods
"""
tsne2 = TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto", perplexity=30)
Z2_tsne = tsne2.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(Z2_tsne)
df_viz_tsne2 = X.copy()
df_viz_tsne2["cluster_z2_tsne"] = labels
print(df_viz_tsne2["cluster_z2_tsne"].value_counts().sort_index())

umap2 = umap.UMAP(n_neighbors=15, min_dist=0.2, n_components=2, random_state=42)
Z2_umap = umap2.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(Z2_umap)
df_viz_umap2 = X.copy()
df_viz_umap2["cluster_z2_umap"] = labels
print(df_viz_umap2["cluster_z2_umap"].value_counts().sort_index())

cluster_z2_tsne
0    477
1    580
2    329
Name: count, dtype: int64


/private/tmp/claude-501/-Users-danielleknutson-MIT-Dropbox-Danielle-Knutson-Dataset-project-DER-data-UROP/bb46b6cd-7152-47fc-a949-36d3b6292711/scratchpad/auditenv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/private/tmp/claude-501/-Users-danielleknutson-MIT-Dropbox-Danielle-Knutson-Dataset-project-DER-data-UROP/bb46b6cd-7152-47fc-a949-36d3b6292711/scratchpad/auditenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster_z2_umap
0    562
1    727
2     97
Name: count, dtype: int64


In [7]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

results = []

pc_options = [2, 3, 5, 8, 10, 12, 15]
k_options = range(2, 9)

for n_pc in pc_options:
    pca = PCA(n_components=n_pc, random_state=42)
    X_pca = pca.fit_transform(X)

    explained_var = pca.explained_variance_ratio_.sum()

    for k in k_options:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X_pca)

        sil = silhouette_score(X_pca, labels)
        ch = calinski_harabasz_score(X_pca, labels)
        db = davies_bouldin_score(X_pca, labels)

        results.append({
            "n_pc": n_pc,
            "k": k,
            "explained_variance": explained_var,
            "silhouette": sil,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
            "inertia": kmeans.inertia_
        })

results_df = pd.DataFrame(results)

ranked_results_df = results_df.copy()
ranked_results_df["silhouette_rank"] = ranked_results_df["silhouette"].rank(ascending=False, method="min")
ranked_results_df["calinski_rank"] = ranked_results_df["calinski_harabasz"].rank(ascending=False, method="min")
ranked_results_df["davies_rank"] = ranked_results_df["davies_bouldin"].rank(ascending=True, method="min")
ranked_results_df["explained_rank"] = ranked_results_df["explained_variance"].rank(ascending=False, method="min")
ranked_results_df["complexity_rank"] = (ranked_results_df["n_pc"] + ranked_results_df["k"]).rank(ascending=True, method="min")

ranked_results_df["composite_rank"] = (
    0.35 * ranked_results_df["silhouette_rank"] +
    0.25 * ranked_results_df["calinski_rank"] +
    0.20 * ranked_results_df["davies_rank"] +
    0.10 * ranked_results_df["explained_rank"] +
    0.10 * ranked_results_df["complexity_rank"]
)

ranked_results_df = ranked_results_df.sort_values(
    ["composite_rank", "silhouette", "calinski_harabasz", "davies_bouldin", "n_pc", "k"],
    ascending=[True, False, False, True, True, True]
).reset_index(drop=True)

best_row = ranked_results_df.iloc[0]
best_n_components = int(best_row["n_pc"])
best_n_clusters = int(best_row["k"])

print(f"Selected PCA components: {best_n_components}")
print(f"Selected number of clusters: {best_n_clusters}")
display(ranked_results_df.head(10))

Selected PCA components: 2
Selected number of clusters: 6


,n_pc,k,explained_variance,silhouette,calinski_harabasz,davies_bouldin,inertia,silhouette_rank,calinski_rank,davies_rank,explained_rank,complexity_rank,composite_rank
0,2,6,0.345898,0.350923,978.801409,0.848554,3690.792545,1.0,1.0,1.0,43.0,9.0,6.00
1,2,4,0.345898,0.345983,949.744874,0.880434,5480.531584,3.0,2.0,3.0,43.0,4.0,6.85
2,2,3,0.345898,0.350736,925.357258,0.942484,7176.493311,2.0,5.0,6.0,43.0,2.0,7.65
3,2,5,0.345898,0.339134,922.855468,0.911761,4568.374197,4.0,6.0,5.0,43.0,6.0,8.80
4,2,7,0.345898,0.336498,948.538158,0.883471,3272.764382,6.0,4.0,4.0,43.0,12.0,9.40
5,2,8,0.345898,0.332228,949.296495,0.878600,2881.958617,7.0,3.0,2.0,43.0,15.0,9.40
6,2,2,0.345898,0.336981,823.377451,1.149495,10520.560742,5.0,7.0,14.0,43.0,1.0,10.70
7,3,3,0.427033,0.283232,612.091675,1.175232,10988.617959,8.0,8.0,15.0,36.0,4.0,11.80
8,3,5,0.427033,0.278675,599.772734,1.041948,7568.048375,10.0,9.0,8.0,36.0,9.0,11.85
9,3,4,0.427033,0.279486,595.346939,1.117927,9036.739900,9.0,11.0,11.0,36.0,6.0,12.30


In [8]:
results_df.to_csv("../outputs/tables/gridsearch_results.csv", index=False)
ranked_results_df.to_csv("../outputs/tables/gridsearch_results_ranked.csv", index=False)

The final PCA/KMeans specification is selected automatically using the grid-search table above.

The notebook ranks each `(n_pc, k)` pair using a weighted composite of:
- higher `silhouette` (higher is better)
- higher `calinski_harabasz`
- lower `davies_bouldin`
- higher `explained_variance`
- lower model complexity (`n_pc + k`)

This keeps the final choice tied to cluster quality while still favoring less complicated solutions when performance is similar. They overall tell you how close together points within a cluster are and how far apart they are from other clusters

In [9]:
def pca_cluster_vis(n_components, n_clusters):
    clustering_result = set_up_pca_kmeans(n_components=n_components, n_clusters=n_clusters)
    pca = clustering_result["pca"]
    df_viz_pca = clustering_result["df_viz"]
    Z = clustering_result["embedding"]
    labels = clustering_result["labels"]
    kmeans = clustering_result["kmeans"]

    loadings = pd.DataFrame(
        pca.components_.T,
        index=X.columns,
        columns=[f"PC{i+1}" for i in range(pca.n_components_)]
    )

    explained = pd.Series(
        pca.explained_variance_ratio_,
        index=loadings.columns,
        name="explained_variance_ratio"
    )
    print("Explained variance ratio:")
    print(explained)

    top_n = 10
    for pc in loadings.columns:
        s = loadings[pc]
        top = s.abs().sort_values(ascending=False).head(top_n).index
        print(f"\nTop {top_n} loadings for {pc}:")
        print(loadings.loc[top, pc].sort_values(key=np.abs, ascending=False))

    top_features = (
        loadings.abs()
        .assign(top_pc=loadings.abs().idxmax(axis=1))
    )

    print("\nTop contributing features to PCA components:")
    display(top_features.sort_values(by=loadings.columns.tolist(), ascending=False).head(10))

    cluster_labels = pd.Series(labels, index=X.index, name="cluster")
    cluster_means = X.assign(cluster=cluster_labels).groupby("cluster").mean()

    ordered_cols = [
        "IOU Utility",
        "POU Utility",
        "Median household income",
        "Median housing value",
        "% Bachelor's+",
        "Poverty rate",
        "% Black",
        "% Hispanic",
        "% Asian",
        "Total population",
        "Log population density",
        "Area in km2",
        "Temp at 2m above ground",
        "Cooling degree days",
        "Heating degree days",
        "Global horizontal irradiance",
        "Mean wind speed at 50 m",
        "Log annual electricity demand",
        "Number of Level 1 chargers",
        "Number of Level 2 chargers",
        "Number of DC fast chargers",
        "Solar PV adoption per 1k pop",
        "EV charger per 1k pop",
        "EV cars per 1k pop",
        "Storage deployment per 1k pop",
        "Wind MW per 100k",
        "Number of turbines per 100k",
    ]
    ordered_cols = [col for col in ordered_cols if col in cluster_means.columns]

    cluster_scale = cluster_means.std(ddof=0).replace(0, np.nan)
    cluster_means_standard = (cluster_means - cluster_means.mean()) / cluster_scale

    print("\nCluster mean feature profiles (standardized):")
    display(cluster_means_standard.round(2))

    plt.figure(figsize=(12, 6))
    sns.heatmap(cluster_means_standard[ordered_cols], cmap="coolwarm", center=0, annot=False)
    plt.title("Cluster-level feature deviations (standardized)")
    plt.xlabel("Feature")
    plt.xticks(rotation=55, horizontalalignment="right")
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.show()

    plot_clusters(
        Z,
        labels,
        centroids=kmeans.cluster_centers_,
        title=f"K-means clusters in PCA space ({n_components} PCs, k={n_clusters})"
    )

    return {
        "pca": pca,
        "embedding": Z,
        "labels": labels,
        "df_viz": df_viz_pca,
        "cluster_means_standard": cluster_means_standard,
    }

best_cluster_result = pca_cluster_vis(best_n_components, best_n_clusters)

cluster_assignments = (
    analysis_df[["zip_code"]]
    .assign(cluster=best_cluster_result["labels"].astype(int))
    .drop_duplicates(subset="zip_code")
    .sort_values("zip_code")
)
cluster_assignments.to_csv("../outputs/tables/pca_kmeans_cluster_assignments.csv", index=False)
display(cluster_assignments.head())

cluster_z_pca
0    175
1    220
2    222
3    175
4    334
5    260
Name: count, dtype: int64
Explained variance ratio:
PC1    0.199128
PC2    0.146770
Name: explained_variance_ratio, dtype: float64

Top 10 loadings for PC1:
% Bachelor's+                 0.341798
Median housing value          0.330035
energy_burden_pct            -0.327673
energy_affordability_index   -0.326743
Median household income       0.311343
Cooling degree days          -0.259752
% Hispanic                   -0.229649
Poverty rate                 -0.226386
% Asian                       0.204376
pct_mobile_home_units        -0.186824
Name: PC1, dtype: float64

Top 10 loadings for PC2:
pct_multifamily_units            0.347080
owner_occupied_rate             -0.344681
Log population density           0.327923
pct_single_family_units         -0.314457
Heating degree days             -0.258685
Total population                 0.246193
Temp at 2m above ground          0.217464
Storage deployment per 1k pop   -0.2014

,PC1,PC2,top_pc
% Bachelor's+,0.341798,0.014870,PC1
Median housing value,0.330035,0.018427,PC1
energy_burden_pct,0.327673,0.077985,PC1
energy_affordability_index,0.326743,0.114616,PC1
Median household income,0.311343,0.105190,PC1
Cooling degree days,0.259752,0.060096,PC1
% Hispanic,0.229649,0.185367,PC1
Poverty rate,0.226386,0.150848,PC1
% Asian,0.204376,0.118544,PC1
pct_mobile_home_units,0.186824,0.141067,PC1



Cluster mean feature profiles (standardized):


,IOU Utility,POU Utility,Number of Level 1 chargers,Number of Level 2 chargers,Number of DC fast chargers,Median household income,Poverty rate,% Bachelor's+,% Black,% Hispanic,...,energy_affordability_index,energy_affordability_gap,Area in km2,Log population density,Solar PV adoption per 1k pop,EV charger per 1k pop,EV cars per 1k pop,Storage deployment per 1k pop,Wind MW per 100k,Number of turbines per 100k
cluster,,,,,,,,,,,,,,,,,,,,,
0,0.66,-0.66,-0.81,-0.26,-0.61,1.72,-1.24,1.27,-0.95,-1.06,...,-1.34,-0.56,-0.47,-0.09,0.16,0.28,1.96,1.67,-0.57,-0.50
1,0.02,-0.02,-0.53,-0.81,-0.41,-1.19,1.70,-1.33,-0.18,1.51,...,1.31,-0.52,1.26,-0.82,1.49,-0.59,-1.09,-0.51,2.13,2.21
2,-2.13,2.13,0.84,0.26,0.08,-0.85,0.92,-0.66,1.82,1.16,...,1.05,-0.09,-0.90,1.12,-1.28,-0.34,-0.82,-1.02,-0.62,-0.55
3,0.49,-0.49,1.83,2.06,1.81,0.79,-0.57,1.35,0.24,-0.90,...,-1.16,2.20,-0.95,1.08,-1.08,2.09,0.10,-0.79,-0.62,-0.55
4,0.90,-0.90,-0.39,-0.31,0.52,0.05,-0.61,-0.03,0.33,-0.04,...,-0.08,-0.41,-0.45,0.37,-0.23,-0.87,-0.46,-0.41,-0.57,-0.52
5,0.07,-0.07,-0.94,-0.94,-1.39,-0.52,-0.20,-0.59,-1.26,-0.66,...,0.22,-0.62,1.51,-1.66,0.94,-0.58,0.30,1.05,0.24,-0.08


/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_8509/1785026761.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_8509/3986231485.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,zip_code,cluster
0,90001,2
1,90002,2
2,90003,2
3,90004,2
4,90005,2


Choropleth map of the selected California ZCTA clusters

In [10]:
zcta_path = Path("../data/raw/boundaries/tl_2023_us_zcta520/tl_2023_us_zcta520.shp")
county_path = Path("../data/raw/boundaries/tl_2023_us_county/tl_2023_us_county.shp")
output_path = Path("../outputs/figures/cluster_choropleth_map.png")
output_path.parent.mkdir(parents=True, exist_ok=True)

zcta_gdf = gpd.read_file(zcta_path)[["ZCTA5CE20", "geometry"]].rename(columns={"ZCTA5CE20": "zip_code"})
zcta_gdf["zip_code"] = zcta_gdf["zip_code"].astype(str).str.zfill(5)

map_gdf = zcta_gdf.merge(cluster_assignments, on="zip_code", how="inner")
cluster_order = sorted(map_gdf["cluster"].unique())
cluster_label_map = {cluster: f"Cluster {cluster + 1}" for cluster in cluster_order}
map_gdf["cluster_label"] = pd.Categorical(
    map_gdf["cluster"].map(cluster_label_map),
    categories=[cluster_label_map[c] for c in cluster_order],
    ordered=True,
)

ca_counties = gpd.read_file(county_path)
if "STATEFP" in ca_counties.columns:
    ca_counties = ca_counties.loc[ca_counties["STATEFP"] == "06"]
ca_counties = ca_counties.to_crs(map_gdf.crs)

palette = sns.color_palette("tab10", n_colors=len(cluster_order))
cmap = mcolors.ListedColormap(palette.as_hex())

fig, ax = plt.subplots(1, 1, figsize=(10.5, 12.5), facecolor="none")
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
map_gdf.plot(
    column="cluster_label",
    categorical=True,
    cmap=cmap,
    linewidth=0.12,
    edgecolor="white",
    legend=True,
    legend_kwds={"title": "Cluster", "loc": "lower left"},
    ax=ax,
)
ca_counties.boundary.plot(ax=ax, color="#475569", linewidth=0.35, alpha=0.65)

xmin, ymin, xmax, ymax = map_gdf.total_bounds
xpad = (xmax - xmin) * 0.03
ypad = (ymax - ymin) * 0.03
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)
ax.axis("off")

legend = ax.get_legend()
if legend is not None:
    legend.set_frame_on(True)
    legend.get_frame().set_edgecolor("#CBD5E1")
    legend.get_frame().set_linewidth(0.8)

plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches="tight", transparent=True)
plt.show()

print(f"Saved choropleth to {output_path}")

Saved choropleth to ../outputs/figures/cluster_choropleth_map.png


/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_8509/1790248294.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
